# Boxes, formats, and IoU

Before any model: the four box formats you will meet, the conversions between them, and the one metric the whole field is built on.

**Runs on:** CPU — about 2 minutes &nbsp;·&nbsp; **Slides:** [Chapter 12 — Object Detection](../../../course-web-slides/ch12/index.html) &nbsp;·&nbsp; **Section:** 01 — The object detection task

---

## Four ways to write the same box

In [ ]:
import numpy as np

# One box around an object, written four ways.
xyxy = np.array([100., 150., 300., 400.])          # x_min, y_min, x_max, y_max
xywh = np.array([100., 150., 200., 250.])          # x_min, y_min, width, height
cxcywh = np.array([200., 275., 200., 250.])        # centre_x, centre_y, w, h
rel_xyxy = xyxy / np.array([640., 480., 640., 480.])   # normalized to [0, 1]

print(f"xyxy      {xyxy}")
print(f"xywh      {xywh}")
print(f"cxcywh    {cxcywh}")
print(f"rel_xyxy  {rel_xyxy.round(4)}")

> ⚠️ **This is where most object-detection bugs live.** COCO uses `xywh`, Pascal VOC uses `xyxy`, YOLO uses normalized `cxcywh`, and none of them will tell you when you have mixed them up — the boxes will simply be in the wrong place.

## Conversions, written once

In [ ]:
def xywh_to_xyxy(b):
    x, y, w, h = b[..., 0], b[..., 1], b[..., 2], b[..., 3]
    return np.stack([x, y, x + w, y + h], axis=-1)

def cxcywh_to_xyxy(b):
    cx, cy, w, h = b[..., 0], b[..., 1], b[..., 2], b[..., 3]
    return np.stack([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2], axis=-1)

def xyxy_to_cxcywh(b):
    x0, y0, x1, y1 = b[..., 0], b[..., 1], b[..., 2], b[..., 3]
    return np.stack([(x0 + x1) / 2, (y0 + y1) / 2, x1 - x0, y1 - y0], axis=-1)

assert np.allclose(xywh_to_xyxy(xywh), xyxy)
assert np.allclose(cxcywh_to_xyxy(cxcywh), xyxy)
assert np.allclose(xyxy_to_cxcywh(xyxy), cxcywh)
print("all round-trips agree")

> **Note** — Write the assertions. They cost nothing and they catch the conversion bug at the moment you introduce it rather than three hours later, in a picture.

## Drawing them

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

rng = np.random.default_rng(0)
img = rng.random((480, 640, 3)) * 0.3 + 0.5

boxes = np.array([[100., 150., 300., 400.],
                  [350., 100., 560., 260.],
                  [420., 300., 600., 450.]])
labels = ["cat", "dog", "chair"]

fig, ax = plt.subplots(figsize=(8, 6))
ax.imshow(img)
for b, lab in zip(boxes, labels):
    x0, y0, x1, y1 = b
    ax.add_patch(patches.Rectangle((x0, y0), x1 - x0, y1 - y0,
                                   fill=False, edgecolor="#ff7a1a", lw=2))
    ax.text(x0, y0 - 6, lab, color="#ff7a1a", fontsize=11, weight="bold")
ax.axis("off"); plt.show()

## IoU: the metric everything rests on

In [ ]:
def iou(a, b):
    """Intersection over union, xyxy format."""
    x0 = np.maximum(a[..., 0], b[..., 0])
    y0 = np.maximum(a[..., 1], b[..., 1])
    x1 = np.minimum(a[..., 2], b[..., 2])
    y1 = np.minimum(a[..., 3], b[..., 3])
    inter = np.clip(x1 - x0, 0, None) * np.clip(y1 - y0, 0, None)
    area_a = (a[..., 2] - a[..., 0]) * (a[..., 3] - a[..., 1])
    area_b = (b[..., 2] - b[..., 0]) * (b[..., 3] - b[..., 1])
    return inter / (area_a + area_b - inter + 1e-9)

truth = np.array([100., 150., 300., 400.])
for name, pred in [("perfect", truth),
                   ("slightly off", truth + 15),
                   ("half overlap", np.array([200., 150., 400., 400.])),
                   ("no overlap", np.array([400., 100., 550., 250.]))]:
    print(f"{name:14s} IoU {iou(pred, truth):.3f}")

Expected output:

```
perfect        IoU 1.000
slightly off   IoU 0.7xx
half overlap   IoU 0.3xx
no overlap     IoU 0.000
```

## Seeing what an IoU threshold means

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(15, 3.4))
for ax, shift in zip(axes, [0, 20, 45, 75, 110]):
    pred = truth + shift
    ax.imshow(img)
    for b, c, lab in [(truth, "#12b886", "truth"), (pred, "#c0392b", "pred")]:
        x0, y0, x1, y1 = b
        ax.add_patch(patches.Rectangle((x0, y0), x1 - x0, y1 - y0,
                                       fill=False, edgecolor=c, lw=2))
    ax.set_title(f"IoU {iou(pred, truth):.2f}", fontsize=11); ax.axis("off")
plt.suptitle("The standard threshold is 0.5. Is that strict enough for you?", y=1.03)
plt.tight_layout(); plt.show()

**IoU 0.5 is a loose box.** It is the field's convention, not a law — and whether it is adequate depends entirely on what the box is used for downstream. A robot gripper needs 0.9; a photo-tagging feature does not.

## Non-maximum suppression

In [ ]:
def nms(boxes, scores, threshold=0.5):
    order = np.argsort(scores)[::-1]
    keep = []
    while len(order):
        i = order[0]
        keep.append(i)
        if len(order) == 1:
            break
        rest = order[1:]
        overlaps = np.array([iou(boxes[i], boxes[j]) for j in rest])
        order = rest[overlaps < threshold]
    return keep

# A detector fires several times on the same object.
raw = np.array([[100., 150., 300., 400.],
                [108., 158., 305., 405.],
                [ 95., 145., 298., 396.],
                [350., 100., 560., 260.]])
scores = np.array([0.92, 0.88, 0.79, 0.85])

kept = nms(raw, scores)
print(f"{len(raw)} raw detections -> {len(kept)} after NMS: {kept}")

fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 5))
for ax, idx, title in [(a1, range(len(raw)), "raw detections"),
                       (a2, kept, "after NMS")]:
    ax.imshow(img)
    for i in idx:
        x0, y0, x1, y1 = raw[i]
        ax.add_patch(patches.Rectangle((x0, y0), x1 - x0, y1 - y0,
                                       fill=False, edgecolor="#ff7a1a", lw=2))
        ax.text(x0, y0 - 6, f"{scores[i]:.2f}", color="#ff7a1a", fontsize=10)
    ax.set_title(title); ax.axis("off")
plt.tight_layout(); plt.show()

Every dense detector produces overlapping duplicates by construction. **NMS is not a refinement — it is part of the output**, and its threshold is a real hyperparameter that trades missed adjacent objects against duplicate boxes.

---

## What to take away

- Four box formats exist and none of them announce themselves. Write the conversions once, with assertions.
- IoU is the metric everything else is defined on top of.
- The conventional 0.5 threshold is looser than it sounds — look at the picture.
- NMS is part of a detector's output, and its threshold is a real trade-off.